In [0]:
%sql
-- ── Agent 3 Table 1: Transformation Registry ───────────────────────────────

CREATE TABLE IF NOT EXISTS learning_dev.s819534.uc3_transformation_registry_b (

    transformation_id           STRING,

    target_fully_qualified_name STRING,
    target_layer                STRING,

    notebook_path               STRING,
    notebook_language           STRING,

    function_name               STRING,

    source_fully_qualified_name STRING,
    transformation_description  STRING,

    checks_enabled              STRING,
    expected_survival_rate      DOUBLE,

    last_reviewed_timestamp     TIMESTAMP,
    last_review_status          STRING,
    last_reviewer               STRING,

    is_active                   BOOLEAN,
    registered_timestamp        TIMESTAMP,
    registered_by               STRING

)
USING DELTA
COMMENT 'Registers Silver transformation notebooks for review by Agent 3.';

In [0]:
%sql
-- ── Agent 3 Table 2: Review Results ──────────────────────────────────────

CREATE TABLE IF NOT EXISTS learning_dev.s819534.uc3_review_results_b (

    review_id                   STRING,
    transformation_id           STRING,

    target_fully_qualified_name STRING,
    review_timestamp            TIMESTAMP,
    triggered_by                STRING,

    overall_status              STRING,
    total_checks_run            INT,
    total_checks_passed         INT,
    total_checks_warned         INT,
    total_checks_critical       INT,

    check_results               STRING,

    ai_summary                  STRING,
    ai_recommended_action       STRING,
    ai_failure_category         STRING,
    ai_confidence_score         DOUBLE,
    ai_model_version            STRING,
    ai_invocation_id            STRING,

    alert_sent                  BOOLEAN,
    alert_sent_timestamp        TIMESTAMP,
    alert_channel               STRING,

    created_timestamp           TIMESTAMP,
    created_by                  STRING

)
USING DELTA
COMMENT 'Stores Agent 3 SQL review results per transformation per review run.';

In [0]:
%sql
SHOW TABLES IN learning_dev.s819534 LIKE 'uc3_*';

database,tableName,isTemporary
s819534,uc3_review_results_b,false
s819534,uc3_transformation_registry_b,false


In [0]:
%sql
INSERT INTO learning_dev.s819534.uc3_transformation_registry_b
VALUES

-- ── Transaction Silver Transformation ─────────────────────────────────────
(
    'txfm-txn-silver-001',
    'learning_dev.s819534.uc2_silver_store_transactions',
    'SILVER',
    '/Users/s819534@8451.com/data_sentinel_barrows/agent2/data_generation/01_generate_silver_gold',
    'PYTHON',
    'transform_raw_to_silver_transactions',
    'learning_dev.s819534.uc1_raw_store_transactions',
    'Transforms raw store transactions to Silver layer. Deduplicates on transaction_id, excludes VOID records, derives null net_amount, excludes null store_id, standardises store_region.',
    '["JOIN_FANOUT", "NULL_FILTER", "AGGREGATION_ORDER", "ROW_COUNT_RATIO"]',
    0.92,
    NULL,
    'NOT_REVIEWED',
    NULL,
    true,
    current_timestamp(),
    'agent3_admin'
),

-- ── Inventory Silver Transformation ───────────────────────────────────────
(
    'txfm-inv-silver-001',
    'learning_dev.s819534.uc2_silver_inventory_movements',
    'SILVER',
    '/Users/s819534@8451.com/data_sentinel_barrows/agent2/data_generation/01_generate_silver_gold',
    'PYTHON',
    'transform_raw_to_silver_inventory',
    'learning_dev.s819534.uc1_raw_inventory_movements',
    'Transforms raw inventory movements to Silver layer. Filters DELETE CDC operations, deduplicates on event_id by latest sequence, removes malformed records, excludes negative quantity_after, excludes null event_id.',
    '["JOIN_FANOUT", "NULL_FILTER", "AGGREGATION_ORDER", "ROW_COUNT_RATIO"]',
    0.88,
    NULL,
    'NOT_REVIEWED',
    NULL,
    true,
    current_timestamp(),
    'agent3_admin'
);

-- Verify
SELECT
    transformation_id,
    SPLIT(target_fully_qualified_name, '\\.')[2] AS target_table,
    function_name,
    expected_survival_rate,
    last_review_status,
    is_active
FROM learning_dev.s819534.uc3_transformation_registry_b;

transformation_id,target_table,function_name,expected_survival_rate,last_review_status,is_active
txfm-txn-silver-001,uc2_silver_store_transactions,transform_raw_to_silver_transactions,0.92,NOT_REVIEWED,true
txfm-inv-silver-001,uc2_silver_inventory_movements,transform_raw_to_silver_inventory,0.88,NOT_REVIEWED,true


In [0]:
%sql
-- COMMAND ----------
-- Add business_date column to uc3_review_results_b

ALTER TABLE learning_dev.s819534.uc3_review_results_b
ADD COLUMN business_date DATE;

-- Verify
DESCRIBE TABLE learning_dev.s819534.uc3_review_results_b;

col_name,data_type,comment
review_id,string,null
transformation_id,string,null
target_fully_qualified_name,string,null
review_timestamp,timestamp,null
triggered_by,string,null
overall_status,string,null
total_checks_run,int,null
total_checks_passed,int,null
total_checks_warned,int,null
total_checks_critical,int,null


In [0]:
# Python backfill — confirmed working based on diagnostic output
# business_date is at check_results[3].business_date for all rows

import json

dbutils.widgets.text("catalog", "learning_dev", "Target Catalog")
dbutils.widgets.text("schema", "s819534", "Target Schema")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA  = dbutils.widgets.get("schema")
TABLE   = f"{CATALOG}.{SCHEMA}.uc3_review_results_b"

print("Starting business_date backfill...")

rows = spark.sql(f"""
    SELECT review_id, check_results
    FROM {TABLE}
    WHERE business_date IS NULL
      AND check_results IS NOT NULL
""").collect()

print(f"Rows to backfill: {len(rows)}")
print()

updated = 0
skipped = 0

for row in rows:
    review_id         = row["review_id"]
    check_results_raw = row["check_results"]
    business_date_found = None

    try:
        check_results = json.loads(check_results_raw)

        for c in check_results:
            if c.get("check_name") == "ROW_COUNT_RATIO":
                bd = c.get("business_date")
                if bd:
                    business_date_found = str(bd)[:10]
                break

    except Exception as e:
        print(f"  Parse error for {review_id[:8]}: {e}")

    if business_date_found:
        spark.sql(f"""
            UPDATE {TABLE}
            SET business_date = CAST('{business_date_found}' AS DATE)
            WHERE review_id   = '{review_id}'
        """)
        print(f"  ✅ {review_id[:8]}...  → business_date = {business_date_found}")
        updated += 1
    else:
        print(f"  ⚠️  {review_id[:8]}...  No ROW_COUNT_RATIO check found")
        skipped += 1

print()
print(f"Backfill complete.")
print(f"  Updated : {updated}")
print(f"  Skipped : {skipped}")

In [0]:
# Verify

print("Verification:")
spark.sql(f"""
    SELECT
        SUBSTRING_INDEX(
            target_fully_qualified_name, '.', -1
        )               AS target_table,
        review_timestamp,
        business_date,
        overall_status,
        total_checks_critical,
        total_checks_warned
    FROM {TABLE}
    ORDER BY review_timestamp DESC
""").show(truncate=False)